In [67]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import scipy.sparse as sp
from torch_geometric.utils import convert
from torch_sparse import SparseTensor
from utils import feature_norm, build_relationship

In [2]:
from data import FairDataset

In [4]:
dataset = 'pokec_z'
device = "cpu"
data = FairDataset(dataset=dataset, device=device)

In [6]:
data.load_data()

In [7]:
data.info()

====================Data Information====================
Dataset: pokec_z
# Nodes: 67796
# Edges: 1303712
# Features: 277
# Classes: 2
# Train samples: 4000
# Val samples: 2565
# Test samples: 2566


In [8]:
X = data.features
Y = data.labels
EI = data.edge_index   # SparseTensor

In [9]:
idx_tr, idx_va, idx_te = data.idx_train, data.idx_val, data.idx_test
in_dim = X.shape[1]
out_dim = 1

In [11]:
features = X
sens = data.sens
A_base = EI
k_per_node = 2
device = "cpu"

In [12]:
X = torch.nn.functional.normalize(features, p=2, dim=1).cpu()
s = sens.cpu().long()
idx0 = (s == 0).nonzero(as_tuple=True)[0]
idx1 = (s == 1).nonzero(as_tuple=True)[0]
if len(idx0) == 0 or len(idx1) == 0:
    raise ValueError("Only one sensitive group present in data.")

In [14]:
features

tensor([[ 1., 14.,  1.,  ...,  0.,  0.,  0.],
        [ 0., 33.,  1.,  ...,  0.,  0.,  0.],
        [ 1., 66.,  1.,  ...,  0.,  0.,  0.],
        ...,
        [ 1., 12.,  0.,  ...,  0.,  0.,  0.],
        [ 1., 47.,  1.,  ...,  0.,  0.,  0.],
        [ 1., 12.,  1.,  ...,  0.,  0.,  0.]])

In [13]:
X

tensor([[0.0336, 0.4698, 0.0336,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7932, 0.0240,  ..., 0.0000, 0.0000, 0.0000],
        [0.0140, 0.9239, 0.0140,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0187, 0.2248, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0211, 0.9919, 0.0211,  ..., 0.0000, 0.0000, 0.0000],
        [0.0284, 0.3413, 0.0284,  ..., 0.0000, 0.0000, 0.0000]])

In [17]:
X[idx0].shape

torch.Size([43962, 277])

In [18]:
X[idx1].shape

torch.Size([23834, 277])

In [21]:
S = X[idx0] @ X[idx1].T
S.shape

torch.Size([43962, 23834])

In [22]:
k = min(k_per_node, S.shape[1])
k

2

In [23]:
topv, topk = torch.topk(S, k=k, dim=1)

In [25]:
topv.shape

torch.Size([43962, 2])

In [26]:
topk.shape

torch.Size([43962, 2])

In [28]:
I = idx0.repeat_interleave(k)
I.shape

torch.Size([87924])

In [36]:
J = idx1[topk.reshape(-1)]
J.shape

torch.Size([87924])

In [42]:
pairs = torch.stack([I, J], dim=0)
pairs.shape

torch.Size([2, 87924])

In [43]:
pairs = torch.unique(pairs, dim=1)
pairs.shape

torch.Size([2, 87924])

In [48]:
base_row, base_col, _ = A_base.coo()
print(base_row.shape, base_col.shape)

torch.Size([1303712]) torch.Size([1303712])


In [55]:
base_set = set(zip(base_row.cpu().tolist(), base_col.cpu().tolist()))
len(base_set)

1303712

In [58]:
keep = []
for a, b in pairs.T.tolist():
    if a != b and (a, b) not in base_set and (b, a) not in base_set:
        keep.append([a, b])
len(keep)

87909

In [59]:
pairs = torch.tensor(keep, dtype=torch.long, device=device).T
pairs.shape

torch.Size([2, 87909])

In [60]:
pairs

tensor([[    0,     0,     2,  ..., 67792, 67794, 67794],
        [50177, 60948, 34949,  ..., 64366, 40997, 47318]])

In [61]:
cand_pairs_ij = pairs
M = cand_pairs_ij.shape[1]
theta0 = torch.full((M,), -2.5)

In [69]:
theta = nn.Parameter(theta0.to(device))
theta.shape

torch.Size([87909])

In [73]:
from models import EdgeAdder

In [74]:
edge_adder = EdgeAdder(data.features.shape[0], cand_pairs_ij, device=device).to(device)

In [82]:
A_ = edge_adder.sparse_tensor()

In [83]:
A_blend = (EI + edge_adder.sparse_tensor())

In [85]:
A_blend.coalesce()

SparseTensor(row=tensor([    0,     0,     0,  ..., 67795, 67795, 67795]),
             col=tensor([    0,     2,     3,  ..., 57955, 61427, 67795]),
             size=(67796, 67796), nnz=1479530, density=0.03%)